<p align="center">
  <img src="https://raw.githubusercontent.com/CSSB-SNU/Thal-Kak_for_release/main/assets/Character.png" height="72">
  &nbsp;&nbsp;
  <img src="https://raw.githubusercontent.com/CSSB-SNU/Thal-Kak_for_release/main/assets/logo_light.png" height="132">
</p>

<a href="https://colab.research.google.com/github/CSSB-SNU/Thal-Kak_for_release/blob/main/Thalkak.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

End-to-end protein structure prediction: **MSA** (ColabFold) → **structure** (Boltz-2 / Chai-1 / Protenix / ESMFold2) → **OpenMM relaxation**, with per-model confidence-based top-5 selection.

**How to run:** set your inputs in step 1, then `Runtime` → `Run all`.

> **Requirements**
> - A **GPU** runtime: `Runtime` → `Change runtime type` → **GPU** (T4 is fine).
> - Protein targets only (RNA needs a large local database built offline).
> - Environment install is fast (**~3 min** via pixi); the first run also downloads model weights (a few minutes).

In [ ]:
#@title 1. Input sequence & options
import os

jobname = "T1201" #@param {type:"string"}
#@markdown - Protein sequence. For a **complex**, separate chains with `:`.
query_sequence = "ETGCNKALCASDVSKCLIQELCQCRPGEGNCSCCKECMLCLGALWDECCDCVGMCNPRNYSDTPPTSKSTVEELHEPIPSLFRALTEGDTQLNWNIVSFPVAEELSHHENLVSFLETVNQPHHQNVSVPSNNVHAPYSSDKEHMCTVVYFDDCMSIHQCKISCESMGASKYRWFHNACCECIGPECIDYGSKTVKCMNCMFGTKHHHHHH" #@param {type:"string"}
#@markdown - Stoichiometry, e.g. `A1` (monomer), `A2` (homodimer), `A1B1`
#@markdown   (heterodimer). Leave as `UNK` to use one copy of each chain.
stoichiometry = "UNK" #@param {type:"string"}
#@markdown ---
#@markdown ### Model & run options
structure_model = "boltz2" #@param ["boltz2", "chai1", "protenix", "esmfold2"]
relax_method = "openmm" #@param ["openmm", "none"]
num_seeds = 5 #@param {type:"integer"}

# Build a FASTA (one record per ":"-separated chain).
seqs = [s.strip().upper().replace(" ", "")
        for s in query_sequence.split(":") if s.strip()]
assert seqs, "query_sequence is empty"
workdir = f"/content/{jobname}"
os.makedirs(workdir, exist_ok=True)
fasta_path = f"{workdir}/{jobname}.fa"
with open(fasta_path, "w") as f:
    for i, s in enumerate(seqs):
        f.write(f">{jobname}_{chr(65 + i)}\n{s}\n")

print(f"Wrote {len(seqs)} chain(s) to {fasta_path}")
print(f"model={structure_model}  relax={relax_method}  "
      f"seeds={num_seeds}  stoi={stoichiometry}")

In [ ]:
#@title Optional: cache model weights to Google Drive
#@markdown Weights download to a **local** cache during the run (reliable).
#@markdown If `drive_cache_dir` already holds a complete cache it is read
#@markdown straight from Drive; otherwise, after a **successful** run the
#@markdown local cache is copied there for reuse. Uncheck to keep weights
#@markdown for this session only.
use_drive_cache = True #@param {type:"boolean"}
drive_cache_dir = "/content/drive/MyDrive/thalkak_cache" #@param {type:"string"}

import os
LOCAL_CACHE = "/content/thalkak_cache"
_CACHE = {"BOLTZ_CACHE": "boltz", "HF_HOME": "hf",
          "CHAI_DOWNLOADS_DIR": "chai", "PROTENIX_CHECKPOINT_DIR": "protenix"}
_cache_writeback = []  # (local, drive) pairs to copy after a successful run
if use_drive_cache:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(drive_cache_dir, exist_ok=True)
    for env_var, sub in _CACHE.items():
        drive_sub = os.path.join(drive_cache_dir, sub)
        local_sub = os.path.join(LOCAL_CACHE, sub)
        if os.path.exists(os.path.join(drive_sub, ".complete")):
            os.environ[env_var] = drive_sub          # complete -> read from Drive
            if env_var == "HF_HOME": os.environ["HF_HUB_OFFLINE"] = "1"
            print(f"{sub}: reading from Drive cache")
        else:
            os.environ[env_var] = local_sub          # download locally this run
            _cache_writeback.append((local_sub, drive_sub))
            print(f"{sub}: local download, will cache to Drive on success")
else:
    for env_var, sub in _CACHE.items():
        os.environ[env_var] = os.path.join(LOCAL_CACHE, sub)
    print("Drive cache off; weights stay in this session only.")

In [ ]:
#@title 2. Clone the Thal-Kak repository
import os, subprocess

REPO = "CSSB-SNU/Thal-Kak_for_release"
REPO_DIR = "/content/Thal-Kak_for_release"

if not os.path.isdir(REPO_DIR):
    rc = subprocess.run(
        ["git", "clone", "--depth", "1",
         f"https://github.com/{REPO}.git", REPO_DIR]
    ).returncode
    if rc != 0:
        raise RuntimeError("git clone failed.")
    print("Cloned", REPO)
else:
    print("Repo already present, skipping clone.")

In [ ]:
#@title 3. Install dependencies (pixi) — ~3 min
#@markdown Installs the [pixi](https://pixi.sh) package manager, resolves the
#@markdown locked `thalkak` environment from `pixi.lock` (no dependency solve),
#@markdown and applies the ColabFold templates patch. Re-running is cheap.
import os, shlex

script = r"""
set -e
export PATH="$HOME/.pixi/bin:$PATH"
if ! command -v pixi >/dev/null 2>&1; then
  echo "== Installing pixi =="
  curl -fsSL https://pixi.sh/install.sh | bash
  export PATH="$HOME/.pixi/bin:$PATH"
fi
cd /content/Thal-Kak_for_release
echo "== Creating the thalkak env from pixi.lock =="
pixi install
echo "== Post-install (ColabFold templates patch) =="
pixi run postinstall
echo "== Install step complete =="
"""

rc = os.system("bash -c " + shlex.quote(script))
if rc != 0:
    raise RuntimeError("Installation failed — see the log above.")

In [ ]:
#@title 4. Run Thal-Kak (MSA → structure → relax)
import subprocess, shlex

cmd = (
    'export PATH="$HOME/.pixi/bin:$PATH" && '
    "cd /content/Thal-Kak_for_release && "
    # run inside the pixi env; -u so progress prints stream live in Colab
    "PYTHONUNBUFFERED=1 pixi run python -u thalkak.py full --msa colab "
    f"--structure {shlex.quote(structure_model)} "
    f"--relax {shlex.quote(relax_method)} "
    f"--seq {shlex.quote(fasta_path)} "
    f"--stoi {shlex.quote(stoichiometry)} "
    f"--n_seed {int(num_seeds)} "
    f"--base_dir {shlex.quote(workdir)}"
)
# Stream stdout/stderr line by line so progress shows live in the cell.
proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait() != 0:
    raise RuntimeError("Thal-Kak run failed — see the log above.")
print("Done. Outputs under:", workdir)

# Success: copy locally-downloaded weights to Drive (complete caches only).
import os, shutil
for local_sub, drive_sub in globals().get("_cache_writeback", []):
    if not os.path.isdir(local_sub) or os.path.exists(os.path.join(drive_sub, ".complete")):
        continue
    try:
        print(f"caching {os.path.basename(local_sub)} to Drive ...", flush=True)
        shutil.copytree(local_sub, drive_sub, dirs_exist_ok=True)
        open(os.path.join(drive_sub, ".complete"), "w").close()
    except Exception as e:
        print(f"  (skipped Drive cache for {os.path.basename(local_sub)}: {e})", flush=True)

In [ ]:
#@title 5. View a predicted structure (per model)
#@markdown Use the dropdowns to switch model / coloring; the view updates live.
import os, glob
import ipywidgets as W
from IPython.display import display
import matplotlib.pyplot as plt, matplotlib as mpl
from matplotlib.colors import Normalize, LinearSegmentedColormap
try:
    import py3Dmol
except ImportError:
    os.system("pip -q install py3Dmol"); import py3Dmol

top5_dirs = glob.glob(f"{workdir}/top5/*_results_*")
assert top5_dirs, "No top5 dir found - did the run finish?"
top5_dir = max(top5_dirs, key=os.path.getmtime)  # newest run (stoi/params may differ)
relaxed_dirs = sorted(glob.glob(f"{top5_dir}/relaxed/*"))
relaxed_dir = relaxed_dirs[0] if relaxed_dirs else None
model_ids = sorted({int(os.path.basename(p).split("_")[1].split(".")[0])
                    for p in glob.glob(f"{top5_dir}/model_*.pdb")}) or [1]

def structure_path(n):
    if relaxed_dir:
        hits = glob.glob(f"{relaxed_dir}/model_{n}_*.pdb")
        if hits: return hits[0]
    p = f"{top5_dir}/model_{n}.pdb"
    return p if os.path.exists(p) else None

def style(color):
    # relaxed/unrelaxed PDBs carry pLDDT in the B-factor column
    if color == "pLDDT":
        return {"cartoon": {"colorscheme": {"prop": "b", "gradient": "rwb",
                                            "min": 40, "max": 100}}}
    return {"cartoon": {"color": "spectrum"}}

model_dd = W.Dropdown(options=model_ids, value=model_ids[0], description="Model:")
color_dd = W.Dropdown(options=["pLDDT", "N to C (rainbow)"], value="pLDDT",
                      description="Color:")
cbar = W.Output()
view = py3Dmol.view(width=700, height=500)

def draw_colorbar(color):
    cbar.clear_output(wait=True)
    with cbar:
        fig, ax = plt.subplots(figsize=(4.5, 0.42))
        if color == "pLDDT":
            # matplotlib "RdBu": 0->red, 1->blue (matches 3Dmol "rwb")
            cmap, norm, label = plt.get_cmap("RdBu"), Normalize(40, 100), "pLDDT"
        else:
            cmap = LinearSegmentedColormap.from_list(
                "roygb", ["red", "orange", "yellow", "green", "blue"])  # 3Dmol spectrum
            norm, label = Normalize(0, 1), "N-term  ->  C-term"
        cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation="horizontal")
        cb.set_label(label)
        if color != "pLDDT":
            cb.set_ticks([])
        plt.show()

def refresh(_=None):
    p = structure_path(model_dd.value)
    if not p:
        return
    # mutate the existing viewer in place (no re-injection -> no blank in Colab)
    view.removeAllModels()
    view.addModel(open(p).read(), "pdb")
    view.setStyle(style(color_dd.value))
    view.zoomTo()
    view.update()
    draw_colorbar(color_dd.value)

display(W.HBox([model_dd, color_dd]))
p0 = structure_path(model_dd.value)
if p0:
    view.addModel(open(p0).read(), "pdb")
    view.setStyle(style(color_dd.value))
    view.zoomTo()
view.show()
display(cbar)
draw_colorbar(color_dd.value)

model_dd.observe(refresh, names="value")
color_dd.observe(refresh, names="value")


In [ ]:
#@title 6. Metrics (MSA depth, pAE, pLDDT, ranking, energy)
import os, glob, yaml, pandas as pd
from IPython.display import display, Image
import ipywidgets as W

top5_dirs = glob.glob(f"{workdir}/top5/*_results_*")
assert top5_dirs, "No result dirs found - did the run finish?"
top5_dir = max(top5_dirs, key=os.path.getmtime)  # newest run
struct_dir = os.path.join(workdir, "structure", os.path.basename(top5_dir))
common = os.path.join(struct_dir, "common")
relaxed_dirs = sorted(glob.glob(f"{top5_dir}/relaxed/*"))
relaxed_dir = relaxed_dirs[0] if relaxed_dirs else None

def _first_png(*pats):
    for pat in pats:
        hits = sorted(glob.glob(pat))
        if hits: return hits[0]
    return None

def show_png(caption, *pats):
    png = _first_png(*pats)
    if png: print(caption); display(Image(png))
    else: print(f"{caption}: image not found.")

def show_msa_depth():
    a3ms = [p for p in glob.glob(f"{workdir}/msa/*/*.a3m") if "_env" not in p]
    if a3ms:
        depth = sum(1 for l in open(a3ms[0]) if l.startswith(">"))
        print(f"MSA depth: {depth} sequences  ({os.path.basename(a3ms[0])})")
    show_png("Coverage", f"{workdir}/msa/*/*coverage*.png")

def show_ranking():
    csv = glob.glob(f"{common}/*results_summary.csv")
    if not csv: print("summary csv not found"); return
    print("Top-5 models are chosen by ranking_score (descending):")
    display(pd.read_csv(csv[0]).head(10))

def show_energy():
    if not relaxed_dir: print("no relaxed dir"); return
    ep = os.path.join(relaxed_dir, "energies.yaml")
    if not os.path.exists(ep): print("energies.yaml not found (relax=none?)"); return
    display(pd.DataFrame(yaml.safe_load(open(ep)) or {}).T)

TABS = [
    ("MSA depth", show_msa_depth),
    ("pAE",       lambda: show_png("pAE", f"{common}/*pae*.png")),
    ("pLDDT",     lambda: show_png("pLDDT", f"{common}/*plddt*.png")),
    ("Ranking",   show_ranking),
    ("Energy",    show_energy),
]
outs = [W.Output() for _ in TABS]
tab = W.Tab(children=outs)
for i, (t, _) in enumerate(TABS): tab.set_title(i, t)
for o, (_, fn) in zip(outs, TABS):
    with o:
        try: fn()
        except Exception as e: print("error:", e)
display(tab)


In [ ]:
#@title 7. Download results
#@markdown Zips the latest run (its structures, top-5, relaxed models,
#@markdown confidence logs) plus the MSA, and downloads it. Optionally copy
#@markdown to Google Drive.
save_to_google_drive = False #@param {type:"boolean"}
import os, glob, zipfile

# Latest run only (same selection as the viewer / metrics cells).
top5_dirs = glob.glob(f"{workdir}/top5/*_results_*")
assert top5_dirs, "No results found - did the run finish?"
top5_dir = max(top5_dirs, key=os.path.getmtime)
run_name = os.path.basename(top5_dir)
roots = [top5_dir,
         os.path.join(workdir, "structure", run_name),
         os.path.join(workdir, "msa")]

zip_path = f"/content/{jobname}.result.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root in roots:
        if not os.path.isdir(root): continue
        for dirpath, _, fnames in os.walk(root):
            for fn in fnames:
                full = os.path.join(dirpath, fn)
                z.write(full, os.path.relpath(full, workdir))
print("Result archive:", zip_path, f"(run: {run_name})")

if save_to_google_drive:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/{jobname}.result.zip"
    shutil.copyfile(zip_path, dst)
    print("Saved to", dst)

from google.colab import files
files.download(zip_path)

## Notes

- **Models & weights.** Each backend downloads its own weights on first run (e.g. `~/.boltz` for Boltz-2, HuggingFace cache for ESMFold2). Weights are distributed under their providers' own terms.
- **Outputs.** For a job named `T1201` the results live under `/content/T1201/`: `msa/`, `structure/`, and `top5/<job>/` with `model_1..5.pdb`, `relaxed/<method>/`, and `method_log.yaml` provenance.
- **RNA/DNA.** This notebook targets proteins. RNA MSA needs a large local database (`prepare_db.sh`) that is impractical to build on Colab — run RNA targets locally instead.
- **License.** Thal-Kak is Apache-2.0; see `LICENSE` and `NOTICE` in the repo.